In [5]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Path & Data Loading
PROJECT_ROOT = Path(r"C:\Users\USER\OneDrive\New folder\Desktop\KENYA_HOUSEHOLD_ELECTRIC_POWER_CONSUMPTION")
parquet_path = PROJECT_ROOT / "cleaned_household_power.parquet"

df = pd.read_parquet(parquet_path)

if not isinstance(df.index, pd.DatetimeIndex):
    df.index = pd.to_datetime(df.index)

df = df.sort_index()

# Identify primary active power column safely
power_col = next(
    (col for col in ["Global_active_power", "global_active_power"] if col in df.columns),
    df.columns[0]
)
df[power_col] = pd.to_numeric(df[power_col], errors="coerce")

# 2. Temporal & Load Regime Features
df["Hour"] = df.index.hour
df["DayOfWeek"] = df.index.dayofweek  # 0=Mon, 6=Sun
df["Weekday_Name"] = df.index.day_name()
df["Month"] = df.index.month
df["Month_Name"] = df.index.strftime("%b")

# Load regimes based on quartiles of active power
df["Load_Regime"] = pd.qcut(
    df[power_col],
    q=4,
    labels=["Q1: Low/Idle", "Q2: Moderate", "Q3: Elevated", "Q4: Peak Draw"]
)

print(f"Loaded shape: {df.shape}")
print(f"Target active power column: '{power_col}'")
df[[power_col, "Hour", "Weekday_Name", "Month_Name", "Load_Regime"]].head()

Loaded shape: (968294, 18)
Target active power column: 'Global_active_power'


,Global_active_power,Hour,Weekday_Name,Month_Name,Load_Regime
DateTime,,,,,
2006-12-16 17:24:00,4.216,17,Saturday,Dec,Q4: Peak Draw
2006-12-16 17:25:00,5.360,17,Saturday,Dec,Q4: Peak Draw
2006-12-16 17:26:00,5.374,17,Saturday,Dec,Q4: Peak Draw
2006-12-16 17:27:00,5.388,17,Saturday,Dec,Q4: Peak Draw
2006-12-16 17:28:00,3.666,17,Saturday,Dec,Q4: Peak Draw


In [ ]:
# --- STEP 2: Central Tendency, Spread, Shape & Robust Summaries ---
s = df[power_col].dropna()

# Central tendency
mean_val = s.mean()
median_val = s.median()
mode_val = s.mode().iloc[0] if not s.mode().empty else np.nan

# Spread & dispersion
min_val = s.min()
max_val = s.max()
range_val = max_val - min_val
q25, q50, q75 = s.quantile([0.25, 0.50, 0.75])
iqr_val = q75 - q25
var_val = s.var(ddof=1)
std_val = s.std(ddof=1)

# Mean Absolute Deviation (around mean and around median)
mad_mean = (s - mean_val).abs().mean()
mad_median = (s - median_val).abs().median()  # True MAD (median absolute deviation)

# Shape & relative variability
skew_val = s.skew()
kurt_excess = s.kurt()          # Fisher (excess kurtosis; normal = 0)
kurt_pearson = kurt_excess + 3  # Pearson kurtosis (normal = 3)
cv_val = std_val / mean_val if mean_val != 0 else np.nan

# Extended quantiles & percentiles
quantiles = s.quantile([0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99])

summary_df = pd.DataFrame({
    "Metric": [
        "Count (Valid)", "Mean", "Median", "Mode (1st)",
        "Min", "Max", "Range (Max - Min)",
        "Q1 (25th Percentile)", "Q2 (50th / Median)", "Q3 (75th Percentile)",
        "IQR (Q3 - Q1)", "Variance (Sample)", "Std Deviation (Sample)",
        "MAD (Mean Abs Dev)", "MAD (Median Abs Dev)",
        "Skewness", "Excess Kurtosis", "Pearson Kurtosis",
        "Coeff. of Variation (CV)", "95th Percentile", "99th Percentile"
    ],
    "Value": [
        len(s), mean_val, median_val, mode_val,
        min_val, max_val, range_val,
        q25, q50, q75,
        iqr_val, var_val, std_val,
        mad_mean, mad_median,
        skew_val, kurt_excess, kurt_pearson,
        cv_val, quantiles.loc[0.95], quantiles.loc[0.99]
    ]
})

summary_df["Value"] = summary_df["Value"].round(4)
print(summary_df.to_string(index=False))